<a href="https://colab.research.google.com/github/jarifrahaman/BERT_fine_tuned_mode/blob/main/Model_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers torch

In [ ]:
from datasets import load_dataset
dataset = load_dataset("dair-ai/emotion")

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  129kB            

split/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
for i in range(20):
  print(dataset['train'][i])

{'text': 'i didnt feel humiliated', 'label': 0}
{'text': 'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake', 'label': 0}
{'text': 'im grabbing a minute to post i feel greedy wrong', 'label': 3}
{'text': 'i am ever feeling nostalgic about the fireplace i will know that it is still on the property', 'label': 2}
{'text': 'i am feeling grouchy', 'label': 3}
{'text': 'ive been feeling a little burdened lately wasnt sure why that was', 'label': 0}
{'text': 'ive been taking or milligrams or times recommended amount and ive fallen asleep a lot faster but i also feel like so funny', 'label': 5}
{'text': 'i feel as confused about life as a teenager or as jaded as a year old man', 'label': 4}
{'text': 'i have been with petronas for years i feel that petronas has performed well and made a huge profit', 'label': 1}
{'text': 'i feel romantic too', 'label': 2}
{'text': 'i feel like i have to make the suffering i m seeing mean something', 'la

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizerFast

SAVE_DIR = "/content/drive/MyDrive/bert-emotion-model"

model = BertForSequenceClassification.from_pretrained(SAVE_DIR)
tokenizer = BertTokenizerFast.from_pretrained(SAVE_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Model loaded on:", device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded on: cpu


In [ ]:
label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]
MAX_LEN = 128

def predict_emotion(text, model=model, tokenizer=tokenizer, device=device):
    model.eval()
    inputs = tokenizer(
        text, padding="max_length", truncation=True,
        max_length=MAX_LEN, return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    idx = int(probs.argmax())
    return label_names[idx], float(probs[idx]), {label_names[i]: float(probs[i]) for i in range(len(label_names))}

In [ ]:
label, conf, scores = predict_emotion("Write the Example Here")
print(f"Predicted: {label} ({conf*100:.1f}%)")
for e, s in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"  {e:10s}: {s*100:5.1f}%")

Predicted: anger (99.9%)
  anger     :  99.9%
  sadness   :   0.0%
  fear      :   0.0%
  love      :   0.0%
  joy       :   0.0%
  surprise  :   0.0%
